# Transfer learning from a pretrained network

Transfer learning allows us to take advantage of representations learned on large datasets and make use of them for learning tasks on unseen data. In this tutorial we keep a ResNet-18 trained on ImageNet frozen, and train a small classical layer together with a quantum model on the features it produces. We perform classification of cats against dogs.

```mermaid
flowchart LR
    A["📷 photo"] --> B["ResNet-18<br/>frozen"]
    B --> C["512 features"]
    C --> D["W: Linear(512 → 5)"]
    D --> E["5 angles"]
    E --> F["QNN(θ)"]
    F --> G["⟨Z₀⟩ … ⟨Z₄⟩"]
    G --> H["head"]
    H --> I["ŷ"]
    F -.->|"parameter shift"| D

    classDef frozen fill:#e8e8e8,stroke:#999999,color:#333333
    classDef trained fill:#dbeafe,stroke:#3b82f6,color:#1e3a8a
    classDef data fill:none,stroke:none
    class B frozen
    class D,F,H trained
    class A,C,E,G,I data
```

Since the frozen network has no trainable parameters, its output is a constant for a given image. The features are computed once, outside any traced function, and from that point on the trained part of the model sees an ordinary array.

We train three parts: the linear layer that reduces the 512 features to the five the register holds, the ansatz parameters $\theta$, and the head that turns the five expectation values into a prediction.

The quantum block does not differentiate under autodiff. We obtain $\partial\langle Z\rangle/\partial x$ and $\partial\langle Z\rangle/\partial\theta$ from the parameter shift rule at the boundary marked above, and backpropagation carries the result into the linear layer.

## Requirements

In addition to Qrisp itself, we make use of the `torchvision` package for the dataset, find below the additional packages required to run this notebook. The versions that this notebook was tested on is also provided.

```
python 3.12
pip install torch==2.4.1 torchvision==0.19.1 certifi optax
```

On macOS the python.org installer ships no CA bundle, so we point OpenSSL at certifi's bundle before downloading.

On first run this downloads the ResNet-18 weights (about 45 MB) and the Oxford-IIIT Pet images (about 800 MB, expanding to roughly 1.6 GB on disk).

In [1]:
import os
import certifi
import numpy as np
import torch
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.datasets import OxfordIIITPet
import matplotlib.pyplot as plt

import jax
import optax
import jax.numpy as jnp

from qrisp import QuantumFloat
from qrisp.operators import Z
from qrisp.qml import Ansatz, Encoding, expectation
from qrisp.jasp import jaspify

os.environ.setdefault("SSL_CERT_FILE", certifi.where())

Matplotlib is building the font cache; this may take a moment.


## The data

Oxford-IIIT Pet ships a binary cat-or-dog label alongside its breed labels. The binary label is the one in `_bin_labels`; `_labels` holds the 37 breeds. One hundred images per class are taken for training, and another hundred from the dataset's own test split are held back.

Cats are labelled `+1` and dogs `-1`, matching the range of the $\langle Z \rangle$ values the circuit returns.

We show two images of each class below, as they come from the dataset.

In [3]:
weights = ResNet18_Weights.DEFAULT
preprocess = weights.transforms()

train_data = OxfordIIITPet(root=".", split="trainval",
                           target_types="binary-category", download=True)
test_data = OxfordIIITPet(root=".", split="test",
                          target_types="binary-category", download=True)


def take(dataset, per_class, seed):
    rng = np.random.default_rng(seed)
    labels = np.array(dataset._bin_labels)
    chosen = np.concatenate([rng.permutation(np.where(labels == c)[0])[:per_class]
                             for c in (0, 1)])
    images = torch.stack([preprocess(dataset[i][0].convert("RGB")) for i in chosen])
    return images, np.where(labels[chosen] == 0, 1.0, -1.0)


per_class = 100
train_images, y_train = take(train_data, per_class, seed=0)
test_images, y_test = take(test_data, per_class, seed=1)

print("train images:", tuple(train_images.shape))

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)>

In [ ]:
labels = np.array(train_data._bin_labels)
shown = np.concatenate([np.where(labels == c)[0][:2] for c in (0, 1)])
names = {0: "cat  (+1)", 1: "dog  (−1)"}

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for ax, i in zip(axes, shown):
    image, label = train_data[i]
    ax.imshow(image.convert("RGB"))
    ax.set_title(names[label])
    ax.axis("off")
fig.tight_layout()

## Freezing the backbone

Replacing the final classification layer with an identity makes the network output its 512 penultimate features instead of class scores. Evaluation mode and `requires_grad_(False)` together mean nothing in the backbone trains.

In [ ]:
backbone = resnet18(weights=weights)
backbone.fc = torch.nn.Identity()
backbone.eval()
for parameter in backbone.parameters():
    parameter.requires_grad_(False)

with torch.no_grad():
    train_features = backbone(train_images).numpy()
    test_features = backbone(test_images).numpy()

print("features        :", train_features.shape)
print("trainable in backbone:", sum(p.requires_grad for p in backbone.parameters()))

## Standardising the features

The reduction from 512 features to the five the register holds is trained rather than fixed, so it belongs to the model and not to this step. Here we only put the features on a common scale, which is ordinary classical work, done once.

We take the mean and standard deviation from the training set alone and apply them to both splits, so nothing about the test images influences the result.

In [ ]:
num_qubits = 5

mean = train_features.mean(axis=0)
std = train_features.std(axis=0) + 1e-8

F_train = jnp.array((train_features - mean) / std)
F_test = jnp.array((test_features - mean) / std)

print("features into the linear layer:", tuple(F_train.shape))

## The model

From here nothing refers to the backbone. The linear layer reduces the standardised features to five rotation angles, the ansatz acts on them, and one observable per qubit gives a vector readout that a small classical layer turns into one prediction.

We keep the angles in $[0, \pi]$ with a sigmoid, so the encoding receives valid rotation angles at every step of training.

In [ ]:
encoding = Encoding.angle(n_features=num_qubits)
ansatz = Ansatz.real_amplitudes(num_qubits=num_qubits, layers=1)


def circuit(x, w):
    qf = QuantumFloat(num_qubits)
    encoding.apply(qf, x)
    ansatz.apply(qf, w)
    return qf


quantum_block = expectation(circuit, [Z(i) for i in range(num_qubits)], precision=0.15)


def angles(f, p):
    return jnp.pi * jax.nn.sigmoid(f @ p["W"] + p["b"])


def head(q, p):
    return jnp.tanh(jnp.dot(q, p["w_out"]) + p["b_out"])


parameters = {
    "W": 0.05 * jax.random.normal(jax.random.PRNGKey(0), (F_train.shape[1], num_qubits)),
    "b": jnp.zeros(num_qubits),
    "theta": jnp.full(ansatz.num_parameters, 0.5),
    "w_out": jnp.ones(num_qubits) / num_qubits,
    "b_out": jnp.array(0.0),
}

print("trainable parameters:", sum(int(v.size) for v in parameters.values()))
print("  linear layer :", int(parameters["W"].size + parameters["b"].size))
print("  ansatz theta :", int(parameters["theta"].size))
print("  head         :", int(parameters["w_out"].size + parameters["b_out"].size))

2581 parameters are trained in total: 2565 in the linear layer, 10 in the ansatz and 6 in the head. The backbone contributes none of them, which is the point of freezing it.

## Training

We obtain the gradient in two pieces. The parameter shift rule gives the derivatives of the expectation values with respect to the five angles and the ten ansatz parameters, each from a pair of circuit evaluations at $\pm\pi/2$. Backpropagation then carries those derivatives through the linear layer, which costs no further circuit evaluations.

This is what keeps the cost independent of the width of the linear layer. One image costs $1 + 2 \times 5 + 2 \times 10 = 31$ circuit evaluations per step whether the layer holds 2565 parameters or far more.

In [ ]:
shift = np.pi / 2
Ex = jnp.eye(num_qubits) * shift
Etheta = jnp.eye(ansatz.num_parameters) * shift


@jaspify(terminal_sampling=True)
def boundary(x, theta):
    """The expectation values and their derivatives, by the parameter shift rule."""
    q = quantum_block(x, theta)
    dq_dx = jnp.stack([(quantum_block(x + Ex[i], theta)
                        - quantum_block(x - Ex[i], theta)) / 2
                       for i in range(num_qubits)])
    dq_dtheta = jnp.stack([(quantum_block(x, theta + Etheta[j])
                            - quantum_block(x, theta - Etheta[j])) / 2
                           for j in range(ansatz.num_parameters)])
    return q, dq_dx, dq_dtheta


@jaspify(terminal_sampling=True)
def q_batch(X, theta):
    """The expectation values for a whole batch of angle vectors."""
    return jnp.stack([quantum_block(X[i], theta) for i in range(X.shape[0])])


def gradient(f, target, p):
    """The loss and its gradient for one image."""
    x, pullback = jax.vjp(lambda params: angles(f, params), p)
    q, dq_dx, dq_dtheta = boundary(x, p["theta"])
    q = np.asarray(q); dq_dx = np.asarray(dq_dx); dq_dtheta = np.asarray(dq_dtheta)

    def loss_from_q(q, params):
        return (head(q, params) - target) ** 2

    value, (g_q, g_head) = jax.value_and_grad(loss_from_q, argnums=(0, 1))(jnp.array(q), p)
    g_angles = pullback(jnp.array(dq_dx @ np.asarray(g_q)))[0]

    return float(value), {"W": g_angles["W"], "b": g_angles["b"],
                          "theta": jnp.array(dq_dtheta @ np.asarray(g_q)),
                          "w_out": g_head["w_out"], "b_out": g_head["b_out"]}

Each image needs the expectation values at the unshifted angles and at $\pm\pi/2$ on every angle and every ansatz parameter. All of those go into a single traced function rather than one call each: the tracing machinery holds on to memory per call, so a loop of separate calls grows without bound while one call per image stays flat.

The pullback is where the two halves meet. It turns the five numbers the parameter shift rule produced into a gradient for every parameter of the linear layer, at no cost in circuit evaluations.

We train with Adam, taken from `optax`, on minibatches of four. A single image already gives a gradient closely aligned with the true one, so a small batch costs little in direction and much less in time.

In [ ]:
def outputs(F, p, idx):
    """One prediction per image, from a single traced call."""
    X = jnp.stack([angles(F[i], p) for i in idx])
    q = np.asarray(q_batch(X, p["theta"]))
    return np.array([float(head(jnp.array(q[k]), p)) for k in range(len(idx))])


def score(F, y, p, idx):
    yh = outputs(F, p, idx)
    return float(np.mean((yh - y[idx]) ** 2)), float(np.mean(np.sign(yh) == y[idx]))


batch_size, steps = 4, 40
sampler = np.random.default_rng(1)
tracked = sampler.choice(F_train.shape[0], 24, replace=False)

# clipped first: Adam would otherwise scale sampling noise in a near-zero gradient up to a full step
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(learning_rate=0.02),
)
p = parameters
opt_state = optimizer.init(p)
history = []

loss, acc = score(F_train, y_train, p, tracked)
print(f"start      loss {loss:.3f}  accuracy {acc:.3f}")

for step in range(steps):
    batch = sampler.choice(F_train.shape[0], batch_size, replace=False)
    grads = None
    for i in batch:
        _, g = gradient(F_train[i], y_train[i], p)
        grads = g if grads is None else jax.tree.map(jnp.add, grads, g)
    grads = jax.tree.map(lambda a: a / batch_size, grads)

    updates, opt_state = optimizer.update(grads, opt_state, p)
    p = optax.apply_updates(p, updates)

    if (step + 1) % 5 == 0:
        loss, acc = score(F_train, y_train, p, tracked)
        history.append((step + 1, loss, acc))
        print(f"step {step + 1:3d}  loss {loss:.3f}  accuracy {acc:.3f}")

## Accuracy

We take the sign of the prediction as the class. One image costs a single circuit evaluation here, with no shift points, so reading the accuracy is far cheaper than a training step.

In [ ]:
print("train accuracy:", score(F_train, y_train, p, np.arange(F_train.shape[0]))[1])
print("test  accuracy:", score(F_test,  y_test,  p, np.arange(F_test.shape[0]))[1])

## What this does and does not give you

The classification is real, and it is the pretrained features that make so small a quantum model enough. The backbone has already done the difficult part, turning pixels into something nearly linearly separable.

The layer that reduces those features to five angles is trained through the quantum block rather than fixed in advance, so the circuit receives angles that were learned for it. The price is 31 circuit evaluations per image per step, set by the five angles and the ten ansatz parameters alone. The 2565 parameters behind them are reached by backpropagation and add nothing to that count.

Expectation values are estimated from samples, so the gradient carries noise and the figures move from run to run. A comparable classical layer on the same features would do about as well, so this tutorial demonstrates that the pattern works, not that the quantum head is preferable.

See also the [hybrid model example](../../reference/Examples/HybridQMLModel.rst) for the parameter-packing pattern, and the [QML tutorial](./QMLtutorial.ipynb) for the primitives themselves.